# 02 · 新项目骨架：一条命令装好，一条命令跑起来

## 这一章在整个项目里的位置

```
00 总览 → 01 存档旧项目 ✅ → [02 新项目骨架 ✅] → 03 自动检查 ✅ → 04 GitHub 与 CI ✅ → 05 准备考题 ✅ → 06 评测集 ✅ → 07 判分 ✅ → 08 MLflow 评测 ✅ → 09 第一个基线 ✅ → 10 第一个工具 ✅ → 11 第二个工具 ✅ → 12 起：更多工具、agent …
```

> **这一章 10 月 7 日已经做完了。** 下面的命令只查看现状，重复运行安全。

## 背景：旧项目为什么"只有你能跑"

换一台电脑、换一个人，旧项目基本跑不起来：

- **装哪些包，有两份清单，还对不上**：`requirements.txt` 里有 torch、没有 pydantic，`pyproject.toml` 里正好反过来
- **代码互相找不到**：88 个文件开头要"手动指路"（`sys.path.insert(...)`），换个文件夹运行就找不到别的代码
- **不该进记录的进了记录**：22 个基因组索引文件

**公司里的做法**：每个 Python 项目都长一个样：一份说明书、一份精确到版本的锁文件、固定的目录、一张"别记"清单。新同事第一天：

```
克隆代码 → uv sync（装好环境）→ uv run raredx（跑起来）
```

两条命令，不用问人。

## 这一章做的 5 件事

| # | 做什么 | 用的技术 | 解决什么问题 |
|---|---|---|---|
| 1 | 开一本记录本 | git | 每次改动都有记录，能退回 |
| 2 | 生成标准骨架 | uv | 结构统一，任何人都知道东西在哪 |
| 3 | 管理依赖 | `pyproject.toml` + `uv.lock` | 换台电脑装出来的版本不一样 |
| 4 | 代码都放进 `src/` | src 布局 | 不再"手动指路" |
| 5 | 列"别记"清单 | `.gitignore` | 密钥、数据、环境不进记录 |

**花费：0。**

---

## 准备

运行下面这一格。它定义一个叫 `run` 的小工具，用来执行命令并显示结果。**不用看代码。**

> 运行 `uv` 的命令时，有时会先出现几行 `Building` / `Installed`：那是 uv 发现代码有变化，顺手更新了一下环境，可以忽略。

In [ ]:
import os, subprocess
from pathlib import Path
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
os.chdir(ROOT)
os.environ.update(NO_COLOR="1", GH_PAGER="cat", GIT_PAGER="cat")
def run(cmd, check=True):
    print(f"$ {cmd}")
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    print(r.stdout + r.stderr)
    if check and r.returncode != 0:
        raise RuntimeError(f"failed: {cmd}")
print("项目目录:", ROOT)

---

## 第 1 步 · 开一本记录本：git

### 在干什么
在新项目的文件夹里开一本记录本，以后每次改动都记一笔。

### 背景
旧项目 5 个月只记了 9 笔，最后两个月的工作根本没记。出了问题，不知道是哪次改动引起的，也退不回去。

### 用了什么技术，为什么

| 词 | 是什么 | 比喻 |
|---|---|---|
| **git** | 记录代码每一次修改的工具 | 实验记录本 |
| **提交**（commit） | 记一笔：改了什么、谁改的、什么时候、为什么 | 记录本上的一页 |
| **`main`** | 正本 | 记录本本身（01 号的分支是复印件） |

记录本本身是项目里一个隐藏的文件夹 `.git/`，不用打开它，git 的命令会替你读写。

**为什么是 git**：所有公司都用，没有第二个选择。

### 公司里什么情况怎么做
- **小步、经常提交**：每做完一小件事就提交一次。攒一周再提交，出了问题就分不清是哪一处
- **提交说明有固定写法**：`类型: 说明`。`feat:` 新功能、`fix:` 修错、`docs:` 文档、`chore:` 杂活。翻历史时一眼就能分清

### 执行
看新项目最早的 3 笔记录：

In [ ]:
run("git log --oneline --reverse | head -3")

### 结果怎么看
- 最上面一行 `3977655 chore: project skeleton` 是新项目的第一笔，就是这一章做的事
- 前面那串 `3977655` 是这一笔的编号，后面是说明

### 面试怎么说
> "Small, frequent commits with conventional messages, so the history explains why every change was made."

---

## 第 2 步 · 生成标准骨架：uv

### 在干什么
用一条命令生成 Python 项目的标准文件。

### 背景
旧项目的结构是一点点"长"出来的，五套代码并存，新人不知道从哪看起。

### 用了什么技术，为什么
**uv**：Python 的项目工具，一个工具管三件事：生成项目、装环境、运行。当时运行的命令（已经做过）：

```bash
uv init --package --name raredx --python 3.12
```

它生成了这几样：

| 生成的 | 作用 | 比喻 |
|---|---|---|
| [`pyproject.toml`](../../pyproject.toml) | 项目叫什么、需要哪些包、入口命令是什么 | 说明书 |
| [`.python-version`](../../.python-version) | 用 Python 3.12 | |
| [`src/raredx/`](../../src/raredx/) | 放全部代码的地方（第 4 步讲） | |

**为什么用 uv 而不是 pip 或 conda**：uv 是现在新 Python 项目的主流选择，速度快、自带锁文件（第 3 步）。conda 在科研里常见，但在公司的软件项目里越来越少用。你电脑上 conda 的环境不受影响：uv 给这个项目单独准备 Python 3.12。

### 公司里什么情况怎么做
- **新项目都用工具或公司模板生成**，不手写这些文件
- **入口命令**：说明书里写好 `raredx = "raredx.cli:app"`，装好之后敲 `raredx` 就能用，不用记"去哪个文件夹运行哪个脚本"

### 执行
看说明书的前 20 行：

In [ ]:
run("head -20 pyproject.toml")

### 结果怎么看
- `name = "raredx"`：项目名
- `dependencies = [...]`：运行时需要的包（httpx、pydantic……，05 号加的）
- `[project.scripts]` 下面的 `raredx = "raredx.cli:app"`：定义了入口命令 `raredx`

文件后面还有 `[tool.ruff]`、`[tool.mypy]` 这些，是 03 号检查工具的设置，到时候再讲。

### 面试怎么说
> "It's a standard uv package: one pyproject.toml declares the project, its dependencies and its CLI entry point."

---

## 第 3 步 · 管理依赖：说明书 + 锁文件

### 在干什么
把项目需要的包和它们的**精确版本**记下来，任何人、任何电脑都能装出一模一样的环境。

### 背景
包每隔几周就出新版本，新版本有时会改变结果，有时直接报错。研究里常见的情况："三个月前能跑，今天装了新版本就跑不通了"，或者"我电脑上是 0.480，同事电脑上是 0.479"。

### 用了什么技术，为什么

| 文件 / 工具 | 作用 | 比喻 |
|---|---|---|
| `pyproject.toml` | 需要哪些包，只写大概的版本范围（比如 `pydantic>=2.13`） | 采购意向 |
| [`uv.lock`](../../uv.lock) | 每一个包的**精确版本和指纹**，包括包依赖的包 | 精确到批号的采购单 |
| `uv sync` | 按锁文件把环境装好 | 照单采购 |
| `.venv/` | 装好的环境，这个项目专用 | 专用实验台 |

包分两组：
- **运行需要的**（pydantic、typer、httpx）：将来部署时会装
- **只在开发时需要的**（ruff、mypy、pytest、jupyter）：写在 `[dependency-groups] dev` 里，部署时不装

**锁文件进记录，`.venv/` 不进**：锁文件几百 KB，别人拿到就能装出一样的环境；`.venv/` 几百 MB，每台电脑自己装。记采购单，不记货。

### 公司里什么情况怎么做
- **加一个新包**：`uv add 包名`。它会同时改说明书和锁文件，两个一起提交
- **新同事第一天**：`uv sync`，环境就和所有人一样
- **自动检查（CI）里**：`uv sync --locked`，锁文件和说明书对不上就直接报错，依赖不能悄悄变

### 执行

In [ ]:
run("uv sync")
run("uv run raredx --help")

### 结果怎么看
- `uv sync`：`Resolved ... packages` 和 `Checked ... packages`，没有报错：环境和锁文件一致
- `raredx --help`：列出可以用的命令，现在有 `data`（05 号加的）和 `eval`（06 号加的）。能看到这些，说明项目装好了、入口命令能用

### 面试怎么说
> "Dependencies are declared in pyproject.toml and pinned in uv.lock, so my laptop, CI and later Docker install exactly the same versions."

---

## 第 4 步 · 代码都放进 src/：不再"手动指路"

### 在干什么
所有代码放在 `src/raredx/` 下面，像正式安装的软件一样：在任何文件夹、任何脚本里，都能直接 `import raredx`。

### 背景
旧项目的脚本，开头常有这样一行：

```python
sys.path.insert(0, str(ROOT))   # 手动告诉 Python："去这个文件夹找代码"
```

88 个文件都这样。换个文件夹运行，或者把文件挪个位置，就找不到别的代码了。

### 用了什么技术，为什么
**src 布局**：代码放在 `src/包名/` 下。`uv sync` 会把这个包"安装"进项目环境，所以脚本、测试、notebook 都用同一句 `import raredx`，不需要知道代码在哪。

现在 `src/raredx/` 里有：

| 文件 | 干什么 |
|---|---|
| `cli.py` | 入口命令 `raredx` |
| `config.py` | 读取设置（05 号讲） |
| `data/` | 数据：下载、核对、整理（05、06 号讲） |
| `evals/` | 评测：考哪些题（06 号）、怎么判分（07 号） |

### 公司里什么情况怎么做
- **逻辑都写在包里，脚本只"调用"**：同一段逻辑不会被抄好几份。旧项目里"调工具"的主循环在 8 个文件里各写了一遍，就是反例
- **测试也通过 `import` 用代码**：测的就是真正会运行的那一份

### 执行
到一个和项目无关的文件夹（你的家目录）里导入 `raredx`：

In [ ]:
run(f'cd ~ && uv run --project "{ROOT}" python -c "import raredx; print(raredx.__file__)"')

### 结果怎么看
打印出 `.../rare-disease-agent/src/raredx/__init__.py`：在家目录里也找到了代码。旧项目做不到这一点。

### 面试怎么说
> "src layout with the package installed into the environment: no sys.path hacks, and the same import works in scripts, tests and notebooks."

---

## 第 5 步 · "别记"清单：.gitignore

### 在干什么
写一张清单，告诉 git 哪些东西永远不要记。

### 背景
有些东西进了记录就很麻烦：
- **密钥**：调大模型要用 API 密钥（相当于银行卡密码）。一旦推到 GitHub，几分钟内就会被自动扫描的机器人发现
- **大文件**：数据进了记录，仓库会变得巨大，而且几乎删不干净。旧项目的 22 个基因组索引文件就是这样
- **装好的环境**：几百 MB，每台电脑都要自己装

### 用了什么技术，为什么
[`.gitignore`](../../.gitignore)：一行写一个不要记的东西。主要几条：

| 不记的 | 为什么 |
|---|---|
| `.env` | 放密钥的文件 |
| `.venv/` | 装好的环境 |
| `/data/` | 下载的数据，用命令重新下载（05 号） |
| `third_party/` | 别人的代码（DeepRare），需要时再下载 |

> **一个真实踩过的坑**：一开始写的是 `data/`，前面没有 `/`。意思就变成了"**任何位置**叫 data 的文件夹都不记"，结果把代码文件夹 `src/raredx/data/` 也挡住了，那次提交少了两个最重要的文件。改成 `/data/`（只挡项目最上层的那个）才好。这件事是 GitHub 的自动检查发现的，04 号细讲。

### 公司里什么情况怎么做
- **新项目第一天就写 `.gitignore`**，不要等出了事再补
- **提交前看一眼要提交哪些文件**（`git status`），确认没有不该有的，也没有漏掉的

### 执行
问 git：这几样东西，分别是被哪一条规则挡住的？

In [ ]:
run("git check-ignore -v .env .venv data third_party")

### 结果怎么看
每一行是 `.gitignore:第几行:规则  被挡住的东西`。比如 `.gitignore:2:.env  .env`：第 2 行的规则 `.env` 挡住了 `.env`。四样都被挡住了。

### 面试怎么说
> "Secrets, data and environments never enter git; data is re-created by a fetch command that verifies checksums."

---

## 本章小结

| 步骤 | 技术 | 公司里的作用 |
|---|---|---|
| 1 记录本 | git | 每次改动都有记录，能退回 |
| 2 骨架 | uv | 每个项目长一个样 |
| 3 依赖 | `pyproject.toml` + `uv.lock` | 谁装都一样 |
| 4 目录 | src 布局 | 不再手动指路 |
| 5 别记清单 | `.gitignore` | 密钥、数据、环境不进记录 |

**一句话**：新项目现在是一个标准的 Python 项目：克隆下来，`uv sync`、`uv run raredx`，两条命令就能跑。

## 面试题

1. **How do you make a Python project reproducible across machines?**
   > "Declare dependencies in pyproject.toml, pin them in a lock file, and install with uv sync — locally, in CI with --locked, and in Docker. Same lock file, same environment everywhere."

2. **What's the src layout and why use it?**
   > "Code lives under src/<package> and is installed into the environment, so imports work the same from scripts, tests and notebooks, and tests exercise the installed package rather than whatever happens to be on the path."

3. **What goes in .gitignore?**
   > "Secrets like .env, virtual environments, caches, data and build outputs. Data is versioned separately and re-created by a fetch step with checksums."

**下一章 03 · 自动检查**：让机器在每次提交前检查代码，有毛病的存不进去。